In [2]:
!pip install --upgrade torchao
!pip uninstall -y transformers accelerate peft
!pip install transformers==5.13.1
!pip install accelerate==1.14.0
!pip install peft==0.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.6 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2

In [3]:
import torch
import transformers
import accelerate
import peft
import torchao

print(torch.__version__)
print(transformers.__version__)
print(accelerate.__version__)
print(peft.__version__)
print(torchao.__version__)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


2.10.0+cu128
5.13.1
1.14.0
0.19.1
0.18.0


In [4]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('smart-mcq-solver-challenge')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/smart-mcq-solver-challenge


In [5]:

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed
from sklearn.model_selection import StratifiedKFold


# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "microsoft/deberta-v3-large"   # bigger backbone, LoRA keeps it trainable
TRAIN_CSV = "/kaggle/input/datasets/chinmayeemilindawale/train-csv/train.csv"
TEST_CSV = "/kaggle/input/datasets/chinmayeemilindawale/csv-test/test.csv"
OUTPUT_DIR = "/kaggle/working/roberta_deberta"
MAX_LEN = 256
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {c: i for i, c in enumerate(OPTIONS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

# target_modules differ by architecture family:
#   DeBERTa-v3 -> "query_proj", "value_proj"
#   BERT/RoBERTa -> "query", "value"
LORA_TARGET_MODULES = ["query_proj", "value_proj"] if "deberta" in MODEL_NAME else ["query", "value"]

USE_WANDB = True
if USE_WANDB:
    import wandb
    wandb.init(project="mcq-science-exam", name="deberta-v3-large-lora")
    

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 26ds2000018 (26ds2000018-iitmaana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:

# -----------------------------
# Load & prep data
# -----------------------------
def load_data():
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df["label"] = train_df["answer"].map(LABEL2ID)
    return train_df, test_df


def to_hf_dataset(df, has_label=True):
    cols = ["id", "prompt"] + OPTIONS + (["label"] if has_label else [])
    return Dataset.from_pandas(df[cols].reset_index(drop=True))


def preprocess(examples, tokenizer):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = [
        [examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [feature.pop(label_name) for feature in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for pred_row, true_label in zip(top3, labels):
        if true_label == pred_row[0]:
            scores.append(1.0)
        elif true_label == pred_row[1]:
            scores.append(0.5)
        elif true_label == pred_row[2]:
            scores.append(1.0 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "map@3": map_at_3(logits, labels)}


In [7]:

def main():
    train_df, test_df = load_data()
    tr_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    set_seed(42)
    base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    # ---- Wrap with LoRA ----
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,   # closest available task type; MC head behaves like a scoring head
        r=16,
        lora_alpha=132,
        lora_dropout=0.1,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["classifier", "pooler"],
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()# sanity check: should show a tiny % of total params


    train_ds = to_hf_dataset(tr_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    val_ds = to_hf_dataset(val_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    test_ds = to_hf_dataset(test_df, has_label=False).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map@3",
        learning_rate=5e-5,              # LoRA typically wants a higher LR than full fine-tune
        per_device_train_batch_size=2,   # smaller since backbone is larger
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=7,              # LoRA often needs a few more epochs to converge
        weight_decay=0.05,
        warmup_ratio=0.1,
        logging_steps=20,
        save_total_limit=1,
        report_to=["wandb"] if USE_WANDB else [],
        fp16=False,
        bf16=False,                      # keep mixed precision off, same instability risk as full fine-tune
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print("Validation results:", trainer.evaluate())

    preds = trainer.predict(test_ds)
    logits = preds.predictions
    top3_idx = np.argsort(-logits, axis=1)[:, :3]
    top3_letters = [" ".join(ID2LABEL[i] for i in row) for row in top3_idx]

    submission = pd.DataFrame({"id": test_df["id"], "prediction": top3_letters})
    submission.to_csv("/kaggle/working/roberta_deberta/submission.csv", index=False)
    print(submission.head())
    print("Saved submission to /kaggle/working/roberta_deberta/submission.csv")

    # save just the LoRA adapter (small, a few MB) rather than the full backbone
    model.save_pretrained("./mcq_lora_adapter")

    if USE_WANDB:
        wandb.finish()


if __name__ == "__main__":
    main()



config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Training Loss,Validation Loss,Epoch,Accuracy,Map@3
3.770045,0.203735,7,0.990000,0.995000


Validation results: {'eval_loss': 0.2037353515625, 'eval_accuracy': 0.99, 'eval_map@3': 0.995}


   id prediction
0   1      A E D
1   2      B C A
2   3      B E D
3   4      E A C
4   5      C D A
Saved submission to /kaggle/working/roberta_deberta/submission.csv


eval/accuracy,▁▄▇█████
eval/loss,█▅▂▁▁▁▁▁
eval/map@3,▁▄▇█████
eval/runtime,██▁▆▆▁▇▇
eval/samples_per_second,▁▁▇▃▃█▂▂
eval/steps_per_second,▁▁▇▃▃█▂▂
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇████
+4,...


In [ ]:
"""ROBERTA

import os
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed
from sklearn.model_selection import StratifiedKFold


# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "roberta-large"
TRAIN_CSV = "/kaggle/input/datasets/chinmayeemilindawale/train-csv/train.csv"
TEST_CSV = "/kaggle/input/datasets/chinmayeemilindawale/csv-test/test.csv"
OUTPUT_DIR = "/kaggle/working/roberta_large"
MAX_LEN = 256
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {c: i for i, c in enumerate(OPTIONS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

LORA_TARGET_MODULES = ["query", "value"]   # RoBERTa/BERT attention layer naming

USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.init(project="mcq-science-exam", name="roberta-large-lora")



# -----------------------------
# Load & prep data
# -----------------------------
def load_data():
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df["label"] = train_df["answer"].map(LABEL2ID)
    return train_df, test_df


def to_hf_dataset(df, has_label=True):
    cols = ["id", "prompt"] + OPTIONS + (["label"] if has_label else [])
    return Dataset.from_pandas(df[cols].reset_index(drop=True))


def preprocess(examples, tokenizer):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = [
        [examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [feature.pop(label_name) for feature in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for pred_row, true_label in zip(top3, labels):
        if true_label == pred_row[0]:
            scores.append(1.0)
        elif true_label == pred_row[1]:
            scores.append(0.5)
        elif true_label == pred_row[2]:
            scores.append(1.0 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "map@3": map_at_3(logits, labels)}


def main():
    train_df, test_df = load_data()
    tr_df, val_df = train_test_split(
        train_df,
        test_size=0.15,
        random_state=42,
        stratify=train_df["label"],
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    set_seed(42)
    base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,   # closest available task type; MC head behaves like a scoring head
        r=16,
        lora_alpha=132,
        lora_dropout=0.1,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["classifier", "pooler"],
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()  # sanity check: should show a tiny % of total params

    train_ds = to_hf_dataset(tr_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    val_ds = to_hf_dataset(val_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    test_ds = to_hf_dataset(test_df, has_label=False).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map@3",
        learning_rate=5e-5,              # LoRA typically wants a higher LR than full fine-tune
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=20,               # LoRA often needs a few more epochs to converge
        weight_decay=0.05,
        warmup_ratio=0.1,
        logging_steps=20,
        save_total_limit=1,
        report_to=["wandb"] if USE_WANDB else [],
        fp16=False,
        bf16=False,                       # keep mixed precision off -- avoids fp16/bf16 instability
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    val_metrics = trainer.evaluate()
    print("Validation results:", val_metrics)

    preds = trainer.predict(test_ds)
    test_logits = preds.predictions

    top3_idx = np.argsort(-test_logits, axis=1)[:, :3]
    top3_letters = [" ".join(ID2LABEL[i] for i in row) for row in top3_idx]
    submission = pd.DataFrame({"id": test_df["id"], "prediction": top3_letters})
    submission.to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)
    print(submission.head())
    print(f"Saved submission to {os.path.join(OUTPUT_DIR, 'submission.csv')}")

    model.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapter"))
    if USE_WANDB:
        wandb.finish()


if __name__ == "__main__":
    main()


"""

In [ ]:
"""
============================================================================
LSTM MCQ Solver — standalone script (no pretrained models)
============================================================================
Trains a from-scratch, Siamese-style bidirectional LSTM to solve the
Smart MCQ Solver Challenge and produces submission.csv.

Architecture:
  - Shared BiLSTM encoder embeds the prompt and each of the 5 options
    (mean-pooled over non-pad timesteps)
  - A small MLP scoring head combines
        [prompt_vec, option_vec, |prompt-option|, prompt*option]
    into a single compatibility score per option
  - Softmax over the 5 scores -> cross-entropy loss against the true label

Install deps:
    pip install torch scikit-learn pandas numpy wandb matplotlib

W&B: this script assumes you're already logged in (wandb login done once
with your token), or set the WANDB_API_KEY env var before running:
    export WANDB_API_KEY=your_token_here

NOTE: written for review, not executed here — check DATA_DIR before running.
============================================================================
"""
'''
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED = 42
DATA_DIR = "/content/drive/MyDrive/smart-mcq-solver"     # <-- update to wherever train.csv/test.csv live
TRAIN_CSV = '/kaggle/input/datasets/chinmayeemilindawale/train-csv/train.csv'#os.path.join(DATA_DIR, "train.csv")
TEST_CSV = '/kaggle/input/datasets/chinmayeemilindawale/csv-test/test.csv'#os.path.join(DATA_DIR, "test.csv")
OUTPUT_DIR = "/kaggle/working/lstm"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, "lstm_confusion_matrix.png")

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {l: i for i, l in enumerate(OPTION_COLS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}

USE_WANDB = True
WANDB_PROJECT = "smart-mcq-solver"
WANDB_ENTITY = None          # set to your W&B username/team if needed

# model / training hyperparams
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LSTM_LAYERS = 1
MAX_PROMPT_LEN = 60
MAX_OPT_LEN = 25
VAL_FRAC = 0.1
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-3
PATIENCE = 4                 # early stopping on val mAP@3
MIN_VOCAB_FREQ = 2
MAX_VOCAB_SIZE = 30000

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================================
# DATA LOADING & CLEANING
# ============================================================================

def load_data(train_csv=TRAIN_CSV, test_csv=TEST_CSV):
    """train.csv: id, prompt, A, B, C, D, E, answer
    test.csv : id, prompt, A, B, C, D, E
    """
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    return train_df, test_df


def clean_text(text) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)              # strip HTML
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # strip URLs
    text = re.sub(r"\s+", " ", text).strip()
    return text


def handle_missing_data(df: pd.DataFrame, text_cols) -> pd.DataFrame:
    df = df.copy()
    for c in text_cols:
        if c in df.columns:
            df[c] = df[c].apply(clean_text)
    all_missing_mask = (df[text_cols] == "").all(axis=1)
    if all_missing_mask.any():
        print(f"Dropping {all_missing_mask.sum()} fully-empty rows")
        df = df[~all_missing_mask].reset_index(drop=True)
    return df


def tokenize_simple(text: str):
    return re.findall(r"[a-zA-Z0-9']+", text.lower())


# ============================================================================
# METRIC — mAP@3
# ============================================================================

def mapk_score(y_true_labels, y_pred_rankings, k=3):
    """y_true_labels: ['B', 'A', ...] | y_pred_rankings: [['A','B','C'], ...]"""
    scores = []
    for true, preds in zip(y_true_labels, y_pred_rankings):
        preds = preds[:k]
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))


def logits_to_top3(logits):
    order = np.argsort(-logits, axis=1)
    return [[IDX2LABEL[j] for j in row[:3]] for row in order]


# ============================================================================
# VOCAB
# ============================================================================

class Vocab:
    """Frequency-based vocabulary built from the train+test corpus (word
    surface forms only — building the vocab from test text does NOT leak
    labels, it just means unseen test words aren't all mapped to <unk>)."""

    def __init__(self, texts, min_freq=MIN_VOCAB_FREQ, max_size=MAX_VOCAB_SIZE):
        counter = Counter()
        for t in texts:
            counter.update(tokenize_simple(t))
        self.itos = ["<pad>", "<unk>"] + [
            w for w, c in counter.most_common(max_size) if c >= min_freq
        ]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, self.unk_id) for tok in tokenize_simple(text)][:max_len]
        if len(ids) < max_len:
            ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.itos)


# ============================================================================
# DATASET
# ============================================================================

class MCQLstmDataset(Dataset):
    def __init__(self, df, vocab, max_prompt_len=MAX_PROMPT_LEN, max_opt_len=MAX_OPT_LEN, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_prompt_len = max_prompt_len
        self.max_opt_len = max_opt_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.vocab.encode(row["prompt"], self.max_prompt_len)
        option_ids = [self.vocab.encode(row[c], self.max_opt_len) for c in OPTION_COLS]
        item = {
            "prompt_ids": torch.tensor(prompt_ids, dtype=torch.long),
            "option_ids": torch.tensor(option_ids, dtype=torch.long),   # [5, max_opt_len]
        }
        if self.has_labels:
            item["label"] = torch.tensor(LABEL2IDX[row["answer"]], dtype=torch.long)
        return item


def lstm_collate(batch):
    out = {
        "prompt_ids": torch.stack([b["prompt_ids"] for b in batch]),
        "option_ids": torch.stack([b["option_ids"] for b in batch]),
    }
    if "label" in batch[0]:
        out["label"] = torch.stack([b["label"] for b in batch])
    return out


# ============================================================================
# MODEL
# ============================================================================

class LSTMEncoder(nn.Module):
    """Shared bidirectional LSTM encoder, mean-pooled over non-pad timesteps
    (more robust than the final hidden state when sequences are padded)."""

    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LSTM_LAYERS, dropout=0.3, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != self.pad_id).float()          # [batch, seq_len]
        emb = self.embedding(x)
        out, _ = self.lstm(emb)                      # [batch, seq_len, hidden*2]
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (out * mask.unsqueeze(-1)).sum(dim=1) / lengths
        return self.dropout(pooled)                  # [batch, hidden*2]


class MCQLstmModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, pad_id=0):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_dim, pad_id=pad_id)
        enc_dim = hidden_dim * 2
        self.scorer = nn.Sequential(
            nn.Linear(enc_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, prompt_ids, option_ids):
        # prompt_ids: [batch, plen] | option_ids: [batch, 5, olen]
        batch_size, n_opts, olen = option_ids.shape
        prompt_vec = self.encoder(prompt_ids)                               # [batch, enc_dim]
        opt_vec = self.encoder(option_ids.view(batch_size * n_opts, olen))
        opt_vec = opt_vec.view(batch_size, n_opts, -1)                      # [batch, 5, enc_dim]

        prompt_exp = prompt_vec.unsqueeze(1).expand(-1, n_opts, -1)         # [batch, 5, enc_dim]
        combined = torch.cat([
            prompt_exp, opt_vec,
            torch.abs(prompt_exp - opt_vec),
            prompt_exp * opt_vec,
        ], dim=-1)                                                          # [batch, 5, enc_dim*4]

        return self.scorer(combined).squeeze(-1)                            # [batch, 5] logits


# ============================================================================
# ERROR ANALYSIS — Macro F1 + confusion matrix
# ============================================================================

def run_error_analysis(df, logits, log_to_wandb=False, save_plot=True):
    from sklearn.metrics import f1_score, confusion_matrix

    top1_pred_idx = np.argmax(logits, axis=1)
    pred_labels = [IDX2LABEL[i] for i in top1_pred_idx]
    true_labels = df["answer"].tolist()

    macro_f1 = f1_score(true_labels, pred_labels, average="macro", labels=OPTION_COLS)
    cm = confusion_matrix(true_labels, pred_labels, labels=OPTION_COLS)

    wrong_mask = np.array(pred_labels) != np.array(true_labels)
    wrong_examples = df.loc[wrong_mask, ["id", "prompt", "answer"]].copy()
    wrong_examples["predicted"] = np.array(pred_labels)[wrong_mask]
    wrong_examples = wrong_examples.head(20)

    if save_plot:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(OPTION_COLS)))
        ax.set_yticks(range(len(OPTION_COLS)))
        ax.set_xticklabels(OPTION_COLS)
        ax.set_yticklabels(OPTION_COLS)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Confusion Matrix — LSTM")
        for i in range(len(OPTION_COLS)):
            for j in range(len(OPTION_COLS)):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im)
        fig.tight_layout()
        fig.savefig(CONFUSION_MATRIX_PATH, dpi=150)
        plt.close(fig)

    if log_to_wandb:
        import wandb
        wandb.log({
            "macro_f1": macro_f1,
            "confusion_matrix_image": wandb.Image(CONFUSION_MATRIX_PATH) if save_plot else None,
            "misclassified_examples": wandb.Table(dataframe=wrong_examples),
        })

    return {
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "wrong_examples": wrong_examples,
    }


# ============================================================================
# TRAIN / EVAL LOOP
# ============================================================================

def run_eval(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in loader:
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            logits = model(prompt_ids, option_ids)
            logits_list.append(logits.cpu().numpy())
            if "label" in batch:
                labels_list.extend(batch["label"].numpy().tolist())
    return np.concatenate(logits_list, axis=0), labels_list


def train_lstm(train_df, test_df):
    from sklearn.model_selection import train_test_split

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    tr_df, val_df = train_test_split(
        train_df, test_size=VAL_FRAC, random_state=SEED, stratify=train_df["answer"]
    )

    # vocab from train+test text (word surface forms only, no label leakage)
    vocab_texts = []
    for df in (train_df, test_df):
        for c in ["prompt"] + OPTION_COLS:
            vocab_texts.extend(df[c].tolist())
    vocab = Vocab(vocab_texts)
    print(f"Vocab size: {len(vocab)}")

    train_ds = MCQLstmDataset(tr_df, vocab)
    val_ds = MCQLstmDataset(val_df, vocab)
    test_ds = MCQLstmDataset(test_df, vocab, has_labels=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lstm_collate)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lstm_collate)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lstm_collate)

    model = MCQLstmModel(len(vocab), pad_id=vocab.pad_id).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    criterion = nn.CrossEntropyLoss()

    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name="lstm-siamese-scratch",
            config={
                "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "lr": LR,
                "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "vocab_size": len(vocab),
            },
        )

    best_val_map3 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(prompt_ids, option_ids)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # LSTMs can explode-gradient
            optimizer.step()
            total_loss += loss.item() * prompt_ids.size(0)

        train_loss = total_loss / len(train_ds)

        val_logits, val_labels_idx = run_eval(model, val_loader, device)
        val_true = [IDX2LABEL[l] for l in val_labels_idx]
        val_rankings = logits_to_top3(val_logits)
        val_map3 = mapk_score(val_true, val_rankings)
        scheduler.step(val_map3)

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_map3={val_map3:.4f}")
        if USE_WANDB:
            import wandb
            wandb.log({
                "epoch": epoch, "train_loss": train_loss, "val_map3": val_map3,
                "lr": optimizer.param_groups[0]["lr"],
            })

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)
    model.to(device)

    # ---- final validation metrics (Macro F1, confusion matrix) --------------
    final_val_logits, _ = run_eval(model, val_loader, device)
    error_metrics = run_error_analysis(val_df, final_val_logits, log_to_wandb=USE_WANDB)
    print(f"Best val mAP@3: {best_val_map3:.4f} | Macro F1: {error_metrics['macro_f1']:.4f}")

    if USE_WANDB:
        import wandb
        wandb.log({"final_val_map3": best_val_map3, "final_val_macro_f1": error_metrics["macro_f1"]})
        wandb.finish()

    # ---- test-set predictions -> submission.csv ------------------------------
    test_logits, _ = run_eval(model, test_loader, device)
    test_rankings = logits_to_top3(test_logits)

    return model, vocab, best_val_map3, error_metrics, test_rankings


def build_submission(test_df, rankings, out_path=SUBMISSION_PATH):
    sub = pd.DataFrame({
        "ID": test_df["id"].values,
        "Prediction": [" ".join(r[:3]) for r in rankings],
    })
    sub.to_csv(out_path, index=False)
    print(f"Saved submission to {out_path}")
    return sub
'''

In [ ]:
# ============================================================================
# MAIN
# ============================================================================
'''
def main():
    train_df, test_df = load_data()
    train_df = handle_missing_data(train_df, ["prompt"] + OPTION_COLS)
    test_df = handle_missing_data(test_df, ["prompt"] + OPTION_COLS)

    model, vocab, best_val_map3, error_metrics, test_rankings = train_lstm(train_df, test_df)

    build_submission(test_df, test_rankings)


if __name__ == "__main__":
    main()
'''    